# 10 - LLM Supervised Fine-Tuning (LoRA)

**Pipeline position:** Step 10 of the ImmunoBiology RAG pipeline (after reranker fine-tuning in Notebook 09, before system evaluation in Notebook 11).

## Purpose

Apply **LoRA (Low-Rank Adaptation)** to fine-tune **Qwen3-8B** on immunology question-answer pairs formatted in ChatML/ShareGPT style. This teaches the LLM to produce concise, domain-accurate answers grounded in retrieved context while preserving its general instruction-following capability.

LLaMA-Factory is used with `template: qwen` (Qwen3's ChatML format with `<|im_start|>` tokens). Flash Attention 2 is **optional** and disabled by default (`flash_attn: false` in `config.yaml`); enable it only on A100-40GB with tight VRAM.

## Learning Objectives

1. Understand the ChatML / ShareGPT data format used by LLaMA-Factory.
2. Build a LLaMA-Factory YAML config and `dataset_info.json` programmatically.
3. Launch LoRA SFT training via subprocess.
4. Parse TensorBoard event logs to reconstruct training curves.
5. Compare pre-trained vs. fine-tuned generation quality (ROUGE-L, BERTScore).

## Key Inputs / Outputs

| Direction | File | Description |
|-----------|------|-------------|
| Input | `data/train/sft_train.jsonl` | SFT training data (ChatML format) |
| Input | `data/train/eval_qa.jsonl` | Held-out evaluation QA pairs |
| Output | `outputs/models/llm_finetuned/` | LoRA adapter checkpoints |
| Output | `outputs/llm_eval/*.png` | Training curves and comparison chart |

## Cell 2 -- Imports

Load all dependencies. LLaMA-Factory is invoked as a subprocess, so it does not need to be importable here -- only `train.train_llm_sft` wrapper functions are needed.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Ensure the project root is on sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import constant
from train.train_llm_sft import (
    build_llamafactory_config,
    build_dataset_info,
    run_llamafactory_training,
    parse_tensorboard_logs,
    plot_sft_curves,
    compare_pre_post_llm,
)

print(f"Project root : {PROJECT_ROOT}")
print(f"SFT data     : {constant.sft_train_path}")
print(f"Eval QA      : {constant.eval_qa_path}")
print(f"Base LLM     : {constant.llm_model_name}")

## Cell 3 -- Inspect SFT Data Format (ChatML)

Each record in the SFT JSONL file follows the **ShareGPT / ChatML** format expected by LLaMA-Factory:

```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

We verify the format, show a sample, and plot turn-length distributions.

In [ ]:
# ------------------------------------------------------------------
# Load and inspect the SFT training data
# ------------------------------------------------------------------
sft_records = []
with open(constant.sft_train_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            sft_records.append(json.loads(line))

print(f"Total SFT training samples: {len(sft_records)}")
print(f"\nSample record (first entry):")
print(json.dumps(sft_records[0], indent=2, ensure_ascii=False)[:800])

# Validate format
valid = sum(1 for r in sft_records if "messages" in r and len(r["messages"]) >= 2)
print(f"\nRecords with valid 'messages' field: {valid}/{len(sft_records)}")

# Length distributions by role
user_lens = []
assistant_lens = []
for r in sft_records:
    for msg in r.get("messages", []):
        word_count = len(msg["content"].split())
        if msg["role"] == "user":
            user_lens.append(word_count)
        elif msg["role"] == "assistant":
            assistant_lens.append(word_count)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_lens, bins=40, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].set_title("User message length (words)")
axes[0].axvline(np.mean(user_lens), color="black", linestyle="--",
                label=f"mean={np.mean(user_lens):.0f}")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(assistant_lens, bins=40, color="mediumseagreen", alpha=0.8, edgecolor="white")
axes[1].set_title("Assistant response length (words)")
axes[1].axvline(np.mean(assistant_lens), color="black", linestyle="--",
                label=f"mean={np.mean(assistant_lens):.0f}")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("SFT Data -- Turn Length Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## Cell 4 -- Build LLaMA-Factory Config

Programmatically generate the YAML config and `dataset_info.json` files that LLaMA-Factory reads at runtime. Key LoRA settings: `r=16`, `alpha=32`, targeting `q_proj` and `v_proj` attention matrices.

In [ ]:
# ------------------------------------------------------------------
# Build config files for LLaMA-Factory
# ------------------------------------------------------------------
OUTPUT_DIR  = constant.llm_finetuned_path
LOGGING_DIR = str(Path(OUTPUT_DIR) / "runs")
EVAL_DIR    = str(Path(constant.BASE_DIR) / "outputs" / "llm_eval")
DATASET_DIR = str(Path(constant.sft_train_path).parent)

# 1. Write dataset_info.json
build_dataset_info(
    sft_file    = constant.sft_train_path,
    dataset_dir = DATASET_DIR,
    dataset_name= "immuno_sft",
)

# 2. Write training YAML config
config_path = build_llamafactory_config(
    output_dir  = OUTPUT_DIR,
    logging_dir = LOGGING_DIR,
    dataset_dir = DATASET_DIR,
    dataset_name= "immuno_sft",
)

print(f"Config written to: {config_path}")
print(f"\n--- Config content ---")
with open(config_path, "r") as f:
    print(f.read())

## Cell 5 -- Launch SFT Training

Launch LLaMA-Factory training as a subprocess. The process streams stdout/stderr to this notebook. Training duration depends on data size and GPU -- expect 30-90 minutes on a single A100 for ~3 epochs.

In [ ]:
# ------------------------------------------------------------------
# Run LLaMA-Factory LoRA SFT
# ------------------------------------------------------------------
exit_code = run_llamafactory_training(config_path)

if exit_code == 0:
    print("\nSFT training completed successfully.")
else:
    print(f"\nSFT training exited with code {exit_code}. Check logs above for errors.")

## Cell 6 -- Parse TensorBoard Logs and Plot Training Curves

LLaMA-Factory logs training/eval loss and learning rate to TensorBoard event files. We parse these into plain arrays and visualize them inline. This reveals convergence speed and potential overfitting.

In [ ]:
# ------------------------------------------------------------------
# Parse TensorBoard logs and visualize
# ------------------------------------------------------------------
history = parse_tensorboard_logs(LOGGING_DIR)

if not history or not history.get("train_steps"):
    print("No TensorBoard logs found. If training has not run yet, run Cell 5 first.")
else:
    print(f"Parsed {len(history['train_steps'])} training steps, "
          f"{len(history.get('eval_steps', []))} eval checkpoints.")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].plot(history["train_steps"], history["train_loss"],
                 alpha=0.5, color="steelblue", linewidth=0.7, label="Train loss")
    if history.get("eval_steps"):
        axes[0].plot(history["eval_steps"], history["eval_loss"],
                     color="tomato", linewidth=2, marker="o", markersize=4, label="Eval loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Cross-entropy Loss")
    axes[0].set_title("SFT Training & Evaluation Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Learning rate schedule
    if history.get("lr_steps"):
        axes[1].plot(history["lr_steps"], history["lr"],
                     color="mediumpurple", linewidth=1.5)
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Learning Rate")
        axes[1].set_title("LR Schedule (Cosine with Warmup)")
        axes[1].grid(alpha=0.3)

    plt.suptitle("LLM SFT Training Curves", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Also save via the library function
    plot_sft_curves(history, EVAL_DIR)
    print(f"Charts saved to: {EVAL_DIR}")

## Cell 7 -- Pre/Post Comparison (ROUGE-L, BERTScore)

Evaluate both the base Qwen3-8B model and the LoRA-adapted checkpoint on the held-out `eval_qa.jsonl` set. ROUGE-L measures lexical overlap with reference answers; BERTScore-F1 captures semantic similarity. A clear lift in both metrics validates that SFT improved domain QA quality.

In [ ]:
# ------------------------------------------------------------------
# Compare base LLM vs fine-tuned LLM
# ------------------------------------------------------------------
finetuned_path = str(Path(OUTPUT_DIR) / "best")

compare_pre_post_llm(
    eval_qa_file        = constant.eval_qa_path,
    base_model_path     = constant.llm_model_name,
    finetuned_model_path= finetuned_path,
    output_dir          = EVAL_DIR,
    sample_n            = 50,
)

# Display the comparison chart inline
comparison_img = Path(EVAL_DIR) / "comparison.png"
if comparison_img.exists():
    from IPython.display import Image, display
    display(Image(filename=str(comparison_img)))
else:
    print("Comparison chart not found -- ensure both models are available.")

## Summary

### Outputs Produced

| File | Description |
|------|-------------|
| `outputs/models/llm_finetuned/` | LoRA adapter weights and configs |
| `outputs/models/llm_finetuned/sft_config.yaml` | LLaMA-Factory training config (reproducibility) |
| `outputs/llm_eval/sft_training_curves.png` | Train/eval loss and LR curves |
| `outputs/llm_eval/comparison.png` | Pre- vs. post-SFT ROUGE-L and BERTScore chart |
| `outputs/llm_eval/comparison.csv` | Numeric comparison results |

### Next Notebook

Proceed to **`11_evaluate.ipynb`** to run the comprehensive end-to-end RAG system evaluation (retrieval + generation + RAGAS + latency).